## Finite Element - Boundary Element Coupling for the London Equation


We solve the problem

\begin{align}
\operatorname{curl}^2 \tilde{H} &= 0 && \text{in }\quad \mathbb{R}^3 \setminus \Omega,\\
\operatorname{curl}^2 \tilde{H} + \tilde{H}&= 0 && \text{in } \quad \Omega,\\
\operatorname{div}\tilde{H} &= 0 && \text{in }\quad \mathbb{R}^3,\\
[\tilde{H}\times \nu] &= -H_{0, ex} \times \nu && \text{on }\quad \partial\Omega,\\
[\operatorname{curl}\tilde{H}\times \nu] &= -\operatorname{curl}H_{0, ex} \times \nu && \text{on }\quad \partial\Omega,
\end{align}
by a
- Galerkin discretization with Nédélec finite elements of order zero in $\Omega$.
- Galerkin discretization with Rao-Wilton-Glisson (Raviart-Thomas) boundary elements of order zero on $\Gamma \coloneqq \partial\Omega$.
- Galerkin discretization with Lagrange finite elements of order one on $\Gamma$.

The weak formulation reads as follows:

$$
\begin{array}{lclclcl}
a_{\mathrm{FEM}}(H, H^{\star}) &+& m_{\Gamma}(\lambda, H^{\star}) &&    &=& 0 & \text{ for all }\ \ H^{\star}\in X_h,\\
c_{\mathrm{BEM}}(H, \lambda^{\star}) &+& v_{\mathrm{BEM}}(\lambda, \lambda^{\star}) &+& b_{\mathrm{mult}}(\varphi, \lambda^{\star}) &=& \ell(\lambda^{\star}) & \text{ for all  }\ \ \lambda^{\star}\in Y_h,\\  
&& b_{\mathrm{mult}}(\lambda, \varphi^{\star}) && &=& 0 & \text{ for all  }\ \ \varphi^{\star}\in Z_h,\\  
\end{array}
$$

where we denoted

$$
\begin{array}{lcl}
a_{\mathrm{FEM}}(H, H^{\star}) &\coloneqq& (\operatorname{curl}H, \operatorname{curl}H^{\star})_{\Omega} + (H, H^{\star})_{\Omega}\\
m_{\Gamma}(\lambda, H^{\star}) &\coloneqq& \langle \lambda,\ H^{\star}\times \nu \rangle_{\Gamma}\\
c_{\mathrm{BEM}}(H, \lambda^{\star}) &\coloneqq& \langle \left(\tfrac{1}{2}\mathbf{I} - \mathbf{K}\right)(H \times \nu), \ \ \lambda^{\star}\rangle_{\Gamma},\\
v_{\mathrm{BEM}}(\lambda, \lambda^{\star}) &\coloneqq& \langle \mathbf{V}\lambda, \ \ \lambda^{\star}\rangle_{\Gamma},\\
b_{\mathrm{mult}}(\varphi, \lambda^{\star})&\coloneqq& \langle \nabla_{\Gamma}\varphi,\ \ \lambda^{\star}\rangle_{\Gamma}
\end{array}
$$


In [ ]:
import dolfin
import bempp_cl.api
import numpy as np


from bempp_cl.api.external import fenics

In [ ]:
import gmsh

gmsh.initialize()
gmsh.model.occ.addSphere(0, 0, 0, 1.0, tag=1)
gmsh.model.occ.synchronize()


# gmsh.option.setNumber("Mesh.CharacteristicLengthMin", 0.1)
gmsh.option.setNumber("Mesh.CharacteristicLengthMax", 0.3)
gmsh.model.mesh.generate(3)



gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
gmsh.write("sphere.msh")
gmsh.finalize()

## Convert with dolfin-convert
!dolfin-convert sphere.msh sphere.xml

In [ ]:
from dolfin import Mesh, XDMFFile

mesh = Mesh("sphere.xml")

# mesh = dolfin.UnitCubeMesh(4, 4, 4)

In [ ]:
fenics_space = dolfin.FunctionSpace(mesh, 'Nedelec 1st kind H(curl)', 1)
trace_space, trace_matrix = fenics.fenics_to_bempp_trace_data(fenics_space)

In [ ]:
# Physical parameters
B_0 = 1.0  # Incident field magnitude


# Create geometry 
grid = trace_space.grid

print(f"Grid: {grid.number_of_elements} elements")



In [ ]:
grid.plot()

In [ ]:
div_space  = bempp_cl.api.function_space(grid, "RWG", 0)
curl_space = bempp_cl.api.function_space(grid, "SNC", 0)
p1_space   = bempp_cl.api.function_space(grid, "P", 1) 


n0 = fenics_space.dim()
n1 = div_space.global_dof_count
n2 = p1_space.global_dof_count

print(f"Div space dimension: {n1}")
print(f"Curl space dimension: {curl_space.global_dof_count}")
print(f"P1 space dimension: {n2}")


In [ ]:
@bempp_cl.api.real_callable
def incident_field_tangential(x, n, domain_index, result):
    B_inc = np.array([0.0, 0.0, B_0 * x[0]])
    result[:] = np.cross(B_inc, n)

@bempp_cl.api.real_callable
def incident_field_curl_tangential(x, n, domain_index, result):
    curlB_inc = np.array([-B_0, 0.0, 0.0])
    result[:] = np.cross(curlB_inc, n)


In [ ]:
# Create grid function for RHS
rhs_fun = bempp_cl.api.GridFunction(div_space, fun=incident_field_tangential, dual_space=curl_space)
print(f"RHS grid function created")



In [ ]:
# Single layer boundary operator
V_op = bempp_cl.api.operators.boundary.maxwell.single_layer(div_space, div_space, curl_space)

# Single layer boundary operator
K_op = bempp_cl.api.operators.boundary.maxwell.double_layer(div_space, div_space, curl_space)


In [ ]:
# Maxwell identity operator (for proper inner product)
identity_p1 = bempp_cl.api.operators.boundary.sparse.identity(
    p1_space, p1_space, p1_space
)


identity = bempp_cl.api.operators.boundary.sparse.identity(
    div_space, div_space, curl_space
)


vector_grad = bempp_cl.api.operators.boundary.sparse._vector_grad_product(
    p1_space, div_space, curl_space
)


In [ ]:
from bempp_cl.api.external.fenics import FenicsOperator
from scipy.sparse.linalg import LinearOperator


# Trace Operator

trace_op = LinearOperator(trace_matrix.shape, lambda x: trace_matrix @ x)

# Trial and Test Functions

u = dolfin.TrialFunction(fenics_space)
v = dolfin.TestFunction(fenics_space)

A_fem = FenicsOperator((dolfin.inner(dolfin.curl(u), dolfin.curl(v)) + dolfin.inner(u, v)) * dolfin.dx)



In [ ]:
# Right Hand Side

rhs_fem = np.zeros(fenics_space.dim())
# rhs_bem = (0.5 * identity - K_op) * rhs_fun 

rhs_bem = rhs_fun 
rhs_p1  = np.zeros(p1_space.global_dof_count, dtype=float)

RHS = np.concatenate([rhs_fem, rhs_bem.projections(curl_space), rhs_p1])


In [ ]:
RHS

## Block operator and Linear System

We assemble the block operator
\begin{equation*}
\mathbf{A}_h = \begin{pmatrix} 
A_h & M_h & 0\\
\left(\tfrac{1}{2}I_h + K_h\right) R_h & V_h & B^T_h \\ 
0 & B_h & 0
\end{pmatrix},
\end{equation*}
where

- $A_h$ is the Galerkin finite-element matrix associated to the $(\mathrm{curl}^2 + 1)$ operator.</br>

- $M_h$ is the Galerkin boundary-domain finite element mass matrix.
- $R_h$ is the trace operator matrix: domain to boundary.
- $V_h$ is the Galerkin boundary element matrix associated to the (vector) single layer operator.
- $B_h^T$ is the Galerkin matrix for the gradient of P1 functions, tested with RT.
- $B_h$ is the weak form of the surface divergence operator.

We solve the linear system

\begin{equation*}
\mathbf{A}_h \mathbf{u}_h =  \mathbf{f}_h,
\end{equation*}

where $$ \mathbf{u}_h = \begin{pmatrix} u_h \\ \lambda_h \\ \varphi_h \end{pmatrix}, \quad \text{ and } \quad \mathbf{f}_h = \begin{pmatrix} 0 \\ f^{\Gamma}_h \\ 0 \end{pmatrix}$$

  


In [ ]:
# Block operator
from bempp_cl.api.assembly.blocked_operator import BlockedDiscreteOperator
from scipy.sparse import csc_matrix

blocks = [[None, None, None], [None, None, None], [None, None, None]]

B = vector_grad.weak_form()

blocks[0][0] = A_fem.weak_form()
blocks[0][1] = -trace_matrix.T * identity.weak_form().to_sparse()
blocks[0][2] = csc_matrix((n0, n2), dtype=np.float64)

blocks[1][0] = (0.5 * identity - K_op).weak_form() * trace_op
blocks[1][1] = V_op.weak_form()
blocks[1][2] = B

blocks[2][0] = csc_matrix((n2, n0), dtype=np.float64)
blocks[2][1] = -B.T
blocks[2][2] = csc_matrix((n2, n2), dtype=np.float64)


A = BlockedDiscreteOperator(np.array(blocks))


In [ ]:
from bempp_cl.api.assembly.discrete_boundary_operator import InverseSparseDiscreteBoundaryOperator

# Define preconditioners

# Inverse of FEM Matrix
P1 = InverseSparseDiscreteBoundaryOperator(blocks[0][0].to_sparse().tocsc())

# P2 = Buffa-Christiansen Mass Matrix Inverse preconditioner

def apply_prec(x):
    """Apply the block diagonal preconditioner"""
    n0 = P1.shape[1]

    res1 = P1.dot(x[:n0])
    res2 = x[n0:]
    return np.concatenate([res1, res2])


p_shape = (P1.shape[0] + n1 + n2, P1.shape[1] + n1 + n2)
P = LinearOperator(p_shape, apply_prec, dtype=np.dtype("float64"))



In [ ]:
A

In [ ]:
RHS

In [ ]:

from scipy.sparse.linalg import gmres

it_count = 0

def count_iterations(x):
    global it_count
    it_count += 1

SOL, info = gmres(A, RHS, M=P, rtol = 1e-6, restart=5000, maxiter= 1, callback=count_iterations, callback_type='pr_norm')

print(f"Number of GMRES iterations: {it_count}")
print(f"GMRES converged to a solution: {info == 0}")

In [ ]:
info

In [ ]:
sol_fem    = SOL[:n0]
sol_bem    = SOL[n0:n0+n1]
sol_lambda = SOL[n0+n1:]

In [ ]:
sol_fem

In [ ]:
sol_bem.shape

In [ ]:
sol_lambda.shape

In [ ]:
sol_fem

In [ ]:
# Plots on the boundary (bempp_cl API)

H = trace_matrix * sol_fem

H_fun = bempp_cl.api.GridFunction(div_space, coefficients=H)

# bempp_cl.api.PLOT_BACKEND = "gmsh"
H_fun.plot()




In [ ]:
# Plots in \Omega (fenics)